In [ ]:
import sys

sys.path.append("..")
import json

import polars as pl

from src.preprocess import extract_relation, reverse_geocode_df, run_cluster, to_csv

/Users/affahrizain/projects/multi-pov-ir/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
entity_df = (
    pl.read_csv("../dataset/csv/entities_new.csv")
    .unique(subset=["wikimedia_url"])
    .with_columns(
        pl.col("poi_name_tags")
        .str.split(",")
        .list.eval(pl.element().filter(pl.element() != ""))
    )
    .with_columns(
        pl.col("nearby_pov_cluster")
        .str.split(",")
        .list.eval(pl.element().filter(pl.element() != ""))
        .list.eval(pl.element().cast(pl.Int64))
    )
)

### Bridge

In [3]:
bridge_entity = entity_df.filter(pl.col("category") == "bridge")

Step 1:
- Convert splatone JSON to CSV
- Reverse geocode to obtain detailed place info

In [4]:
data = json.loads(open("../dataset/splatone/jp-bridge.json", "r").read())
bridge_df = reverse_geocode_df(to_csv(data))
bridge_df = bridge_df.filter(pl.col("country_code") == "JP")

Reverse geocoding batches: 100%|██████████| 1/1 [00:00<00:00,  1.72batch/s]


Step 2:
- Clustering and merge similar ones based on spatial and lexical

In [5]:
ner_labels = ["bridge"]
bridge_spot, bridge_cluster = run_cluster(
    bridge_df, ner_labels, similarity_threshold=0.35
)
bridge_cluster = bridge_cluster.with_columns(
    pl.col("entities").list.eval(pl.element().filter(pl.element() != ""))
)

/Users/affahrizain/projects/multi-pov-ir/.venv/lib/python3.13/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
NER: 100%|██████████| 5682/5682 [02:40<00:00, 35.40it/s]


In [31]:
bridge_spot.with_columns(pl.col("entities").list.join(",")).write_csv("../dataset/csv/spots/bridge_spots.csv")

Step 3:
- Extract relation between POI (entities) and POV (spots) (also based on spatial and lexical)

In [6]:
bridge_entity = extract_relation(
    bridge_entity, bridge_cluster, similarity_threshold=0.35
)

Step 4:
- Update entity csv (act as db lookup during retrieval)

In [ ]:
entity_df = entity_df.update(
    bridge_entity.select(["wikimedia_url", "nearby_pov_cluster"]), on="wikimedia_url"
)
entity_df.with_columns(
    [
        pl.col("poi_name_tags").list.join(","),
        pl.col("nearby_pov_cluster").cast(pl.List(pl.String)).list.join(","),
    ]
).sort("entity_id").write_csv("../dataset/csv/entities.csv")